# Local file system - Rust

All 10 Rust examples from [docs/local.md](https://platob.github.io/yggdryl/local/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and expect the
[evcxr](https://github.com/evcxr/evcxr) kernel. Declare the crate once, in
a cell of your own, before running them:

```rust
:dep yggdryl = { version = "0.1", features = ["parquet", "iceberg"] }
```

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::local::File;

let path = std::env::temp_dir().join(format!("yggdryl-doc-lead-{}.bin", std::process::id()));

let mut file = File::create(&path)?;
file.write_all_bytes(b"AAPL")?;
file.flush()?;

assert_eq!(file.read_all_bytes()?, b"AAPL");

drop(file);
let _ = std::fs::remove_file(&path);

## The three roles

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::local::{File, Folder, Path};
use yggdryl::IOKind;

let root = std::env::temp_dir().join(format!("yggdryl-doc-roles-{}", std::process::id()));
let _ = std::fs::remove_dir_all(&root);
std::fs::create_dir_all(root.join("nested"))?;
std::fs::write(root.join("a.bin"), b"a")?;

// A container: it holds no bytes of its own, only children.
let folder = Folder::new(&root)?;
assert_eq!(folder.size(), 0);
assert_eq!(folder.ls(false, false)?.len(), 2);

// A leaf: bytes addressed by offset.
let leaf = File::new(root.join("a.bin"))?;
assert_eq!(leaf.read_all_bytes()?, b"a");

// A location: it answers by looking at what is actually there.
assert_eq!(Path::new(&root)?.kind(), IOKind::Directory);
assert_eq!(Path::new(root.join("a.bin"))?.kind(), IOKind::File);

let _ = std::fs::remove_dir_all(&root);

## Laziness

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::local::{File, Folder};

let root = std::env::temp_dir().join(format!("yggdryl-doc-lazy-{}", std::process::id()));
let _ = std::fs::remove_dir_all(&root);

// Constructing touches nothing.
let folder = Folder::new(&root)?;
let mut leaf = File::new(root.join("nested").join("trades.bin"))?;
assert!(!folder.exists());
assert!(!leaf.exists());

// Reading something absent yields nothing - and still creates nothing.
assert!(folder.ls(true, false)?.is_empty());
assert!(leaf.read_all_bytes()?.is_empty());
assert_eq!(leaf.size(), 0);
assert!(!root.exists());

// Writing creates the file and every missing parent.
leaf.write_all_bytes(b"trade")?;
leaf.flush()?;
assert!(leaf.exists());
assert_eq!(leaf.read_all_bytes()?, b"trade");

drop(leaf);
let _ = std::fs::remove_dir_all(&root);

## A write decides an undecided location

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::local::Path;
use yggdryl::IOKind;

let root = std::env::temp_dir().join(format!("yggdryl-doc-decide-{}", std::process::id()));
let _ = std::fs::remove_dir_all(&root);
std::fs::create_dir_all(&root)?;

// Nothing is there, so nothing has decided what it is.
let mut location = Path::new(root.join("trades.bin"))?;
assert_eq!(location.kind(), IOKind::Unknown);

// A byte write settles it: an undecided location becomes a file.
location.write_all_bytes(b"AAPL")?;
location.flush()?;
assert_eq!(location.kind(), IOKind::File);
assert_eq!(location.read_all_bytes()?, b"AAPL");

// To settle it the other way, say so before writing.
let container = Path::new(root.join("day=2026-08-16"))?;
assert_eq!(container.kind(), IOKind::Unknown);
container.as_directory()?.create()?;
assert_eq!(container.kind(), IOKind::Directory);

drop(location);
let _ = std::fs::remove_dir_all(&root);

## Walking the tree

In [ ]:
use yggdryl::generic::Holder;
use yggdryl::io::IOBase;
use yggdryl::local::Folder;

let root = std::env::temp_dir().join(format!("yggdryl-doc-walk-{}", std::process::id()));
let _ = std::fs::remove_dir_all(&root);

let folder = Folder::new(&root)?;
folder.create()?;

// A child is a handle; writing through it creates the leaf.
let mut leaf = folder.child_by_path("trades.arrows")?;
leaf.write_all_bytes(b"payload")?;
leaf.flush()?;
assert!(matches!(leaf, Holder::File(_)));

// A nested child creates its parent directory on write.
let mut nested = folder.child_by_path("sub/inner.bin")?;
nested.write_all_bytes(b"deep")?;
nested.flush()?;

// Listings are sorted, so two runs agree; recursion reaches the nested leaf.
let names: Vec<String> = folder
    .ls(false, false)?
    .iter()
    .filter_map(|entry| entry.url().and_then(|url| url.file_name()).map(str::to_string))
    .collect();
assert_eq!(names, ["sub", "trades.arrows"]);
assert_eq!(folder.ls(true, false)?.len(), 3);

// A leaf's parent is the directory holding it.
let parent = leaf.parent().expect("a file has a parent");
assert!(parent.is_container());
assert_eq!(parent.url().unwrap(), folder.url());

drop(leaf);
drop(nested);
let _ = std::fs::remove_dir_all(&root);

## The mapping

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::local::File;

let path = std::env::temp_dir().join(format!("yggdryl-doc-growth-{}.bin", std::process::id()));

let mut file = File::create(&path)?;
file.pwrite(0, b"trade")?;

// Writing past the mapping remaps at a larger capacity instead of failing.
let bulk = vec![7_u8; 256 * 1024];
file.append(&bulk)?;
assert_eq!(file.size(), 5 + bulk.len() as u64);
assert!(file.capacity() >= file.size());

// Flushing publishes the logical length, so the file is the bytes, not the mapping.
file.flush()?;
assert_eq!(std::fs::metadata(&path)?.len(), file.size());

drop(file);
let _ = std::fs::remove_file(&path);

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::local::File;

let path = std::env::temp_dir().join(format!("yggdryl-doc-gap-{}.bin", std::process::id()));

let mut file = File::create(&path)?;
file.pwrite(0, b"ab")?;
file.pwrite(5, b"z")?;

// The gap the offset created is zero-filled.
assert_eq!(file.read_all_bytes()?, b"ab\0\0\0z");

drop(file);
let _ = std::fs::remove_file(&path);

## The SIGBUS hazard

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::local::File;

let path = std::env::temp_dir().join(format!("yggdryl-doc-snapshot-{}.bin", std::process::id()));
std::fs::write(&path, b"trade")?;

// The handle - and its mapping - is gone by the time the copy returns.
let mut snapshot = Buffer::new();
File::new(&path)?.copy_into(&mut snapshot)?;

assert_eq!(snapshot.into_bytes(), b"trade");
let _ = std::fs::remove_file(&path);

## A remote backend is a sibling module

In [ ]:
use yggdryl::io::{Buffer, IOBase};
use yggdryl::local::File;

fn head(handle: &dyn IOBase) -> yggdryl::Result<Vec<u8>> {
    handle.read_range(0, 4)
}

let path = std::env::temp_dir().join(format!("yggdryl-doc-agnostic-{}.bin", std::process::id()));

let mut file = File::create(&path)?;
file.write_all_bytes(b"AAPL,100")?;

let memory = Buffer::from_bytes(b"AAPL,100".to_vec());
assert_eq!(head(&file)?, b"AAPL");
assert_eq!(head(&file)?, head(&memory)?);

drop(file);
let _ = std::fs::remove_file(&path);

## Private entries

In [ ]:
use yggdryl::io::IOBase;
use yggdryl::local::Folder;
use yggdryl::Url;

let root = std::env::temp_dir().join("yggdryl-doc-private");
std::fs::create_dir_all(root.join(".git"))?;
std::fs::write(root.join("trades.arrows"), b"x")?;

let folder = Folder::new(&root)?;
assert_eq!(folder.ls(false, false)?.len(), 1);
assert_eq!(folder.ls(false, true)?.len(), 2);

// The rule is one accessor on the location itself, because every child has one.
assert!(Url::from_str("file:///project/.git")?.is_private());
assert!(!Url::from_str("file:///project/trades.arrows")?.is_private());

std::fs::remove_dir_all(&root)?;